## Semantically-Assisted Normal Distributions Transform

In [1]:
import numpy as np
import plotly.graph_objects as go

from tqdm import tqdm
from copy import copy
from typing import Callable, cast
from IPython.display import clear_output

from se_ndt.SemanticVoxel import SemanticVoxel
from se_ndt.LabeledPoint import LabeledPoint
from ndt.Parameters import Parameters
from se_ndt.SemanticReferencePointCloud import SemanticReferencePointCloud
from se_ndt.SemanticTargetPointCloud import SemanticTargetPointCloud
from se_ndt.SemanticOptimization import SemanticOptimization

from mesh.MeshSampler import MeshSampler

import utils.mat_ops as mat
import utils.plotting as plot
import utils.aftr as aftr

Initialize the mesh for sampling and alignment.

In [ ]:
mesh = MeshSampler( 'mesh/meshes/f15_model.obj', 'F-15_scaled', rotation_matrix = np.array( [[1, 0, 0], [0, 0, 1], [0, 1, 0]] ) )
labeled_vertices = mesh.get_labeled_vertices()
mesh.show_mesh()

Initialize `Parameters`, which follows the definition of $\vec{p}$ below, as well as the `ReferencePointCloud`, `TargetPointCloud`, and `Optimization`. We assume the value of $\vec{p}$ begins at identity orientation at the sensor origin. After alignment, $-\vec{p} \equiv P_{sensor}^{target}$.

In [ ]:
p = Parameters( se3 = np.eye(4) )
ref_pc = SemanticReferencePointCloud( y = np.array( labeled_vertices['points'] ), labels = labeled_vertices['labels'] )
tar_pc = SemanticTargetPointCloud( p, ref_pc.get_voxel )
opt = SemanticOptimization()

ref_pc_pts = ref_pc.get_pc_list()
ref_pc_vox = [f'Voxel {i}' for i in range( len( ref_pc_pts ) )]
mesh.display_point_clouds( ref_pc_pts, ref_pc_vox, "Voxelized Reference Point Cloud" )


ref_pc_pts_labeled, ref_pc_labels_labeled = ref_pc.get_pc_list_by_label()
mesh.display_point_clouds( ref_pc_pts_labeled, ref_pc_labels_labeled, "Segmented Reference Point Cloud" )

point_cloud = 2

### OPTION 2:  IMPORT AN AFTR VIRTUAL LIDAR SAMPLE ###
match( point_cloud ):
    case 1:
        aftr_dict = aftr.from_aftr_frame( 'aftr/f-15_model.txt' )
        aftr_dict = aftr.organize_aftr_frame_by_part( aftr_dict )
        tar_pc_pts = aftr_dict['points']
        tar_pc_lbs = aftr_dict['part_labels']
        for i, pt_set in enumerate( tar_pc_pts ):
            for pt in pt_set:   tar_pc.add( LabeledPoint( pt.reshape(( 3, 1 )), tar_pc_lbs[i], ref_pc.get_voxel( pt.reshape(( 3, 1 )) ) ) )
        mesh.display_point_clouds( tar_pc_pts, tar_pc_lbs, "Target Point Cloud" )

    case 2:
        aftr_dict = aftr.from_aftr_frame( 'aftr/lidar_frame.txt' )

        # Approximate transform to center
        R = mat.get_dcm( -2, -10, 11 )
        aftr_dict['points'] = ( R @ aftr_dict['points'].T + np.array( [-9.529, -2.63, -1.518] ).reshape(( 3, 1 )) ).T

        aftr_dict = aftr.organize_aftr_frame_by_part( aftr_dict )
        tar_pc_pts = aftr_dict['points']
        tar_pc_lbs = aftr_dict['part_labels']
        for i, pt_set in enumerate( tar_pc_pts ):
            for pt in pt_set:   tar_pc.add( LabeledPoint( pt.reshape(( 3, 1 )), tar_pc_lbs[i], ref_pc.get_voxel( pt.reshape(( 3, 1 )) ) ) )
        mesh.display_point_clouds( tar_pc_pts, tar_pc_lbs, "Target Point Cloud" )


Initializing SemanticVoxel...
	hstab:  2371 points found
	vstab:  1014 points found
	wing:  7588 points found
	engine:  5327 points found
	fuselage:  8234 points found
Initializing SemanticVoxel...
	vstab:  2040 points found
Initializing SemanticVoxel...
	hstab:  2417 points found
	vstab:  1328 points found
	wing:  7481 points found
	engine:  5121 points found
	fuselage:  8378 points found
Initializing SemanticVoxel...
	vstab:  2737 points found
Initializing SemanticVoxel...
	wing:  6 points found
	engine:  3243 points found
	fuselage:  6380 points found
Initializing SemanticVoxel...
	fuselage:  657 points found
Initializing SemanticVoxel...
	wing:  20 points found
	engine:  3421 points found
	fuselage:  5699 points found
Initializing SemanticVoxel...
	fuselage:  410 points found


Definitions:
- $\vec{p}$:  the parameters to be estimated $\rightarrow (x, y, z, r_x, r_y, r_z)$
- $\textbf{x}$:  the $n \times 3$ matrix of points, $\vec{x}$, contained in the target (sensed) point cloud
- $\textbf{x}'$:  the $n \times 3$ matrix of points, $\vec{x}$ contained in the target point cloud after transformation via the parameters, $p$, where $\textbf{x}' = T(\textbf{x}, p)$
- $pr(\vec{x}_i')$:  the probability of a single point given the alignment, $p$
- $s$:  the total score, or sum of all $pr(\vec{x}_i)$
- $\textbf{y}$:  the $n \times 3$ matrix of points, $\vec{y}$, contained in the reference point cloud
- $\Sigma$:  the covariance of $\textbf{y}$
- $\vec{\mu}$:  the mean of $\textbf{y}$

The normal distribution transform utilizes Newton's method to minimize the score of its cost function, $pr(\textbf{x}')$, by varying the parameters which, in the 3D case, are the translation components $(x, y, z)$ and Euler angles $(r_x, r_y, r_z)$. The cost function is a Gaussian approximation of the negative log likelihood function:

\begin{align*}
pr(\vec{x}) &= d_1 * exp( -\frac{d_2}{2}(\vec{x} - \vec{\mu})^T \Sigma^{-1} (\vec{x} - \vec{\mu}) ) \\\\
score &= - \Sigma_{i = 1}^n pr(\vec{x}_i)
\end{align*}

where

\begin{align*}
q &= \frac{1}{n} \Sigma_i x_i \\\\
\Sigma &= \frac{1}{n - 1} \Sigma_i (x_i - q)(x_i - q)^T
\end{align*}

Depending on the implementation, $d_1$ and $d_2$ can take many forms. In Peter Biber's initial implementation (2003), $d_1 = d_2 = 1$. In Zaganidis' semantically-assisted implementation (2018), $d_2 = 1$ and $d_1 = \frac{1}{\sqrt{(2 \pi)^3 |\Sigma|}}$. Magnusson's implementation is spelled out in equation 6.8 of his dissertation, where he attempts to emulate the mixed negative log-likelihood + uniform distribution function, but poorly defines solving for the constants, $c_1$ and $c_2$. Here, the use Zaganidis' implementation is an optional parameter (default is $d_1 = 1$), which improves the weighting of high-confidence areas over Biber's implementation. In practice, it also tends to apply more weight to dense areas of the reference point cloud. Therefore,

\begin{align*}
pr(\vec{x}) &= \frac{1}{\sqrt{(2 \pi)^3 |\Sigma|}} exp( -\frac{(\vec{x} - \vec{\mu})^T \Sigma^{-1} (\vec{x} - \vec{\mu})}{2} ) \\\\
-- &or --\\\\
pr(\vec{x}) &= exp( -\frac{(\vec{x} - \vec{\mu})^T \Sigma^{-1} (\vec{x} - \vec{\mu})}{2} ) \\\\
s &= - \Sigma_{i = 1}^n pr(\vec{x}_i)
\end{align*}

In code, $s$ is computed by calling a `.get_score()` in the specific instance of `Voxel`. To get the score of the entire point cloud, `Optimization` takes the target point cloud and computes the negative sum of `.get_score()`. The plots below show the cost surface of the Zaganidis cost function as a function of pairs of parameters.

In [ ]:
x_r: tuple[float, float] = ( -2.5, 2.5 )
x_n: float = 51
y_r: tuple[float, float] = ( -2.5, 2.5 )
y_n: float = 51

x = np.linspace( x_r[0], x_r[1], x_n )
y = np.linspace( y_r[0], y_r[1], y_n )

samples = np.zeros( ( y_n, x_n ) )
for i in tqdm( range( x_n ) ):
    for j in range( y_n ):
        tar_pc.set_pose( np.array( [ x[i], y[j], 0, 0, 0, 0 ] ).reshape(( 6, 1 )) )
        samples[i, j] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, x_r, y_r, title = "Zaganidis cost surface over x-y translation", z_label = "Cost" ).show()

y_idx, x_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ x[x_idx], y[y_idx], 0, 0, 0, 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [00:37<00:00,  1.36it/s]


In [ ]:
x_r: tuple[float, float] = ( -2.5, 2.5 )
x_n: float = 51
z_r: tuple[float, float] = ( -2.5, 2.5 )
z_n: float = 51

x = np.linspace( x_r[0], x_r[1], x_n )
z = np.linspace( z_r[0], z_r[1], z_n )

samples = np.zeros( ( z_n, x_n ) )
for i in tqdm( range( x_n ) ):
    for j in range( z_n ):
        tar_pc.set_pose( np.array( [ x[i], 0, z[j], 0, 0, 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, x_r, z_r, title = "Zaganidis cost surface over x-z translation", y_label = "z", z_label = "Cost" ).show()

z_idx, x_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ x[x_idx], 0, z[z_idx], 0, 0, 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [00:39<00:00,  1.30it/s]


In [ ]:
y_r: tuple[float, float] = ( -2.5, 2.5 )
y_n: float = 51
z_r: tuple[float, float] = ( -2.5, 2.5 )
z_n: float = 51

y = np.linspace( y_r[0], y_r[1], y_n )
z = np.linspace( z_r[0], z_r[1], z_n )

samples = np.zeros( ( z_n, y_n ) )
for i in tqdm( range( y_n ) ):
    for j in range( z_n ):
        tar_pc.set_pose( np.array( [ 0, y[i], z[j], 0, 0, 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, y_r, z_r, title = "Zaganidis cost surface over y-z translation", x_label = "y", y_label = "z", z_label = "Cost" ).show()

z_idx, y_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, y[y_idx], z[z_idx], 0, 0, 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [00:39<00:00,  1.29it/s]


In [ ]:
rx_r: tuple[float, float] = ( -np.pi, np.pi )
rx_n: float = 101
ry_r: tuple[float, float] = ( -np.pi, np.pi )
ry_n: float = 101

rx = np.linspace( rx_r[0], rx_r[1], rx_n )
ry = np.linspace( ry_r[0], ry_r[1], ry_n )

samples = np.zeros( ( ry_n, rx_n ) )
for i in tqdm( range( rx_n ) ):
    for j in range( ry_n ):
        tar_pc.set_pose( np.array( [ 0, 0, 0, rx[i], ry[j], 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, rx_r, ry_r, title = "Zaganidis cost surface over roll-pitch rotation", x_label = "roll", y_label = "pitch", z_label = "Cost" ).show()

ry_idx, rx_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, 0, 0, rx[rx_idx], ry[ry_idx], 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 101/101 [02:44<00:00,  1.63s/it]


In [ ]:
rx_r: tuple[float, float] = ( -np.pi, np.pi )
rx_n: float = 101
rz_r: tuple[float, float] = ( -np.pi, np.pi )
rz_n: float = 101

rx = np.linspace( rx_r[0], rx_r[1], rx_n )
rz = np.linspace( rz_r[0], rz_r[1], rz_n )

samples = np.zeros( ( rz_n, rx_n ) )
for i in tqdm( range( rx_n ) ):
    for j in range( rz_n ):
        tar_pc.set_pose( np.array( [ 0, 0, 0, rx[i], 0, rz[j] ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, rx_r, rz_r, title = "Zaganidis cost surface over roll-yaw rotation", x_label = "roll", y_label = "yaw", z_label = "Cost" ).show()

rz_idx, rx_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, 0, 0, rx[rx_idx], 0, rz[rz_idx] ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 101/101 [02:40<00:00,  1.59s/it]


In [ ]:
ry_r: tuple[float, float] = ( -np.pi, np.pi )
ry_n: float = 101
rz_r: tuple[float, float] = ( -np.pi, np.pi )
rz_n: float = 101

ry = np.linspace( ry_r[0], ry_r[1], ry_n )
rz = np.linspace( rz_r[0], rz_r[1], rz_n )

samples = np.zeros( ( rz_n, ry_n ) )
for i in tqdm( range( ry_n ) ):
    for j in range( rz_n ):
        tar_pc.set_pose( np.array( [ 0, 0, 0, 0, ry[i], rz[j] ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, ry_r, rz_r, title = "Zaganidis cost surface over pitch-yaw rotation", x_label = "pitch", y_label = "yaw", z_label = "Cost" ).show()

rz_idx, ry_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, 0, 0, 0, ry[ry_idx], rz[rz_idx] ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

  0%|          | 0/101 [00:00<?, ?it/s]

100%|██████████| 101/101 [02:42<00:00,  1.61s/it]


In [ ]:
x_r: tuple[float, float] = ( -2.5, 2.5 )
x_n: float = 51
rx_r: tuple[float, float] = ( -np.pi, np.pi )
rx_n: float = 101

x = np.linspace( x_r[0], x_r[1], x_n )
rx = np.linspace( rx_r[0], rx_r[1], rx_n )

samples = np.zeros( ( rx_n, x_n ) )
for i in tqdm( range( x_n ) ):
    for j in range( rx_n ):
        tar_pc.set_pose( np.array( [ x[i], 0, 0, rx[j], 0, 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, x_r, rx_r, title = "Zaganidis cost surface over x-roll translation-rotation", x_label = "x", y_label = "roll", z_label = "Cost" ).show()

rx_idx, x_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ x[x_idx], 0, 0, rx[rx_idx], 0, 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [01:13<00:00,  1.44s/it]


In [ ]:
y_r: tuple[float, float] = ( -2.5, 2.5 )
y_n: float = 51
rx_r: tuple[float, float] = ( -np.pi, np.pi )
rx_n: float = 101

y = np.linspace( y_r[0], y_r[1], y_n )
rx = np.linspace( rx_r[0], rx_r[1], rx_n )

samples = np.zeros( ( rx_n, y_n ) )
for i in tqdm( range( y_n ) ):
    for j in range( rx_n ):
        tar_pc.set_pose( np.array( [ 0, y[i], 0, rx[j], 0, 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, y_r, rx_r, title = "Zaganidis cost surface over y-roll translation-rotation", x_label = "y", y_label = "roll", z_label = "Cost" ).show()

rx_idx, y_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, y[y_idx], 0, rx[rx_idx], 0, 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [01:16<00:00,  1.51s/it]


In [ ]:
z_r: tuple[float, float] = ( -2.5, 2.5 )
z_n: float = 51
rx_r: tuple[float, float] = ( -np.pi, np.pi )
rx_n: float = 101

z = np.linspace( z_r[0], z_r[1], z_n )
rx = np.linspace( rx_r[0], rx_r[1], rx_n )

samples = np.zeros( ( rx_n, z_n ) )
for i in tqdm( range( z_n ) ):
    for j in range( rx_n ):
        tar_pc.set_pose( np.array( [ 0, 0, z[i], rx[j], 0, 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, z_r, rx_r, title = "Zaganidis cost surface over z-roll translation-rotation", x_label = "z", y_label = "roll", z_label = "Cost" ).show()

rx_idx, z_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, 0, z[z_idx], rx[rx_idx], 0, 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [01:20<00:00,  1.57s/it]


In [ ]:
x_r: tuple[float, float] = ( -2.5, 2.5 )
x_n: float = 51
ry_r: tuple[float, float] = ( -np.pi, np.pi )
ry_n: float = 101

x = np.linspace( x_r[0], x_r[1], x_n )
ry = np.linspace( ry_r[0], ry_r[1], ry_n )

samples = np.zeros( ( ry_n, x_n ) )
for i in tqdm( range( x_n ) ):
    for j in range( ry_n ):
        tar_pc.set_pose( np.array( [ x[i], 0, 0, 0, ry[j], 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, x_r, ry_r, title = "Zaganidis cost surface over x-pitch translation-rotation", x_label = "x", y_label = "pitch", z_label = "Cost" ).show()

ry_idx, x_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ x[x_idx], 0, 0, 0, ry[ry_idx], 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [01:14<00:00,  1.46s/it]


In [ ]:
y_r: tuple[float, float] = ( -2.5, 2.5 )
y_n: float = 51
ry_r: tuple[float, float] = ( -np.pi, np.pi )
ry_n: float = 101

y = np.linspace( y_r[0], y_r[1], y_n )
ry = np.linspace( ry_r[0], ry_r[1], ry_n )

samples = np.zeros( ( ry_n, y_n ) )
for i in tqdm( range( y_n ) ):
    for j in range( ry_n ):
        tar_pc.set_pose( np.array( [ 0, y[i], 0, 0, ry[j], 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, y_r, ry_r, title = "Zaganidis cost surface over y-pitch translation-rotation", x_label = "y", y_label = "pitch", z_label = "Cost" ).show()

ry_idx, y_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, y[y_idx], 0, 0, ry[ry_idx], 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

  0%|          | 0/51 [00:00<?, ?it/s]

100%|██████████| 51/51 [01:17<00:00,  1.53s/it]


In [ ]:
z_r: tuple[float, float] = ( -2.5, 2.5 )
z_n: float = 51
ry_r: tuple[float, float] = ( -np.pi, np.pi )
ry_n: float = 101

z = np.linspace( z_r[0], z_r[1], z_n )
ry = np.linspace( ry_r[0], ry_r[1], ry_n )

samples = np.zeros( ( ry_n, z_n ) )
for i in tqdm( range( z_n ) ):
    for j in range( ry_n ):
        tar_pc.set_pose( np.array( [ 0, 0, z[i], 0, ry[j], 0 ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, z_r, ry_r, title = "Zaganidis cost surface over z-pitch translation-rotation", x_label = "z", y_label = "pitch", z_label = "Cost" ).show()

ry_idx, z_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, 0, z[z_idx], 0, ry[ry_idx], 0 ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [01:22<00:00,  1.62s/it]


In [ ]:
x_r: tuple[float, float] = ( -2.5, 2.5 )
x_n: float = 51
rz_r: tuple[float, float] = ( -np.pi, np.pi )
rz_n: float = 101

x = np.linspace( x_r[0], x_r[1], x_n )
rz = np.linspace( rz_r[0], rz_r[1], rz_n )

samples = np.zeros( ( rz_n, x_n ) )
for i in tqdm( range( x_n ) ):
    for j in range( rz_n ):
        tar_pc.set_pose( np.array( [ x[i], 0, 0, 0, 0, rz[j] ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, x_r, rz_r, title = "Zaganidis cost surface over x-yaw translation-rotation", x_label = "x", y_label = "yaw", z_label = "Cost" ).show()

rz_idx, x_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ x[x_idx], 0, 0, 0, 0, rz[rz_idx] ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [01:13<00:00,  1.43s/it]


In [ ]:
y_r: tuple[float, float] = ( -2.5, 2.5 )
y_n: float = 51
rz_r: tuple[float, float] = ( -np.pi, np.pi )
rz_n: float = 101

y = np.linspace( y_r[0], y_r[1], y_n )
rz = np.linspace( rz_r[0], rz_r[1], rz_n )

samples = np.zeros( ( rz_n, y_n ) )
for i in tqdm( range( y_n ) ):
    for j in range( rz_n ):
        tar_pc.set_pose( np.array( [ 0, y[i], 0, 0, 0, rz[j] ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, y_r, rz_r, title = "Zaganidis cost surface over y-yaw translation-rotation", x_label = "y", y_label = "yaw", z_label = "Cost" ).show()

rz_idx, y_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, y[y_idx], 0, 0, 0, rz[rz_idx] ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [01:18<00:00,  1.54s/it]


In [ ]:
z_r: tuple[float, float] = ( -2.5, 2.5 )
z_n: float = 51
rz_r: tuple[float, float] = ( -np.pi, np.pi )
rz_n: float = 101

z = np.linspace( z_r[0], z_r[1], z_n )
rz = np.linspace( rz_r[0], rz_r[1], rz_n )

samples = np.zeros( ( rz_n, z_n ) )
for i in tqdm( range( z_n ) ):
    for j in range( rz_n ):
        tar_pc.set_pose( np.array( [ 0, 0, z[i], 0, 0, rz[j] ] ).reshape(( 6, 1 )) )
        samples[j, i] = opt.f( tar_pc.get_points() )

plot.plot_sampled_surface( samples, z_r, rz_r, title = "Zaganidis cost surface over z-yaw translation-rotation", x_label = "z", y_label = "yaw", z_label = "Cost" ).show()

rz_idx, z_idx = np.unravel_index( np.argmin( samples ), samples.shape )
tar_pc.set_pose( np.array( [ 0, 0, z[z_idx], 0, 0, rz[rz_idx] ] ).reshape(( 6, 1 )) )

# gmin_tar_pc_pts = ( tar_pc.p.get_dcm() @ tar_pc_pts.T + tar_pc.p.get_position() ).T

# mesh.display_point_clouds( ref_pc_pts + [ gmin_tar_pc_pts ], ref_pc_vox + list( tar_pc_lbs ), "Sensed and reference point clouds after alignment" )

100%|██████████| 51/51 [01:21<00:00,  1.61s/it]


With the cost function defined, the next step is to compute the Jacobian and Hessian matrices for the optimization. Magnusson loosely refers to two different Jacobians and two different Hessians with inconsistent notation. The equations with consistent notation applied are spelled out below.

- $\textbf{J}_E$:  the first derivative of the rotation defined by the 6D parameter function, $\vec{p}$, defined in eq. 6.19-20 in Magnusson's dissertation
- $\textbf{H}_E$:  the second derivate of the rotation defined by the 6D parameter function, $\vec{p}$, defined in eq. 6.20-21 ub Magnusson's dissertation
- $\textbf{g}$:  the first derivative of the score function, $s$, defined in eq. 6.12 in Magnusson's dissertation
- $\textbf{H}$:  the second derivative of the score function, $s$, defined in eq. 6.13 in Magnusson's dissertation

$\textbf{J}_E$ and $\textbf{H}_E$ are computed per-point and used as $\frac{\delta \vec{x}'_k}{\delta p_i}$ and $\frac{\delta^2 \vec{x}'_k}{\delta p_i \delta p_j}$ in the equations for $g$ and $H$.

A damped implementation of Newton's Method and a Levenberg-Marquardt implementation are included in the `Optimization` class, both solving the 6D minization function:

\begin{align*}
\vec{p}_{n + 1} = \vec{p}_n - H^{-1}g
\end{align*}